#Haystack Question-Answering Framework

**September 2023 update by Denis Rothman**: The goal of introducing Haystack in *Transformers for NLP, 2nd Edition, Chapter 11, Let Your Data DO the Talking: Story, Questions and Answers*, was to *introduce the reader to several platforms beyond using only one set of tools*.

Haystack is an interesting platform to explore for Q&A, and more. However, there are conflicts with Google Colab. You might have to install all the necessary packages locally.

The alternative is [QA.ipynb](https://github.com/Denis2054/Transformers-for-NLP-2nd-Edition/blob/main/Chapter11/QA.ipynb), a Hugging Face implementation of Q&A.
_________________________________________________________________
Former Notebook resources

Notebook Author: [Malte Pietsch](https://www.linkedin.com/in/maltepietsch/)

[Deepset AI Haystack GitHub Repository](https://github.com/deepset-ai/haystack/)


In [1]:
# # Install Haystack
# !pip install farm-haystack==0.6.0

# # Install specific versions of urllib and torch to avoid conflicts with preinstalled versions on Colab
# !pip install urllib3==1.25.4
# !pip install torch==2.0.0+cu118 -f https://download.pytorch.org/whl/torch_stable.html


# Extractive QA in a closed domain (single text)

In [9]:
# # Load a  local model or any of the QA models on Hugging Face's model hub (https://huggingface.co/models)
# from haystack.reader.farm import FARMReader

# reader = FARMReader(model_name_or_path="deepset/roberta-base-squad2", use_gpu=True, no_ans_boost=0, return_no_answer=False)


# # Create document which the model should scan for answers.
# from haystack import Document

# text = "The traffic began to slow down on Pioneer Boulevard in Los Angeles, making it difficult to get out of the city. However, WBGO was playing some cool jazz, and the weather was cool, making it rather pleasant to be making it out of the city on this Friday afternoon. Nat King Cole was singing as Jo and Maria slowly made their way out of LA and drove toward Barstow. They planned to get to Las Vegas early enough in the evening to have a nice dinner and go see a show."
# doc = Document(text=text)

# Install once: pip install haystack-ai
from haystack.components.readers import ExtractiveReader
from haystack.dataclasses import Document

# Load the QA model
reader = ExtractiveReader(model="deepset/roberta-base-squad2")
reader.warm_up()

# Create the document which the model should scan for answers
text = "The traffic began to slow down on Pioneer Boulevard in Los Angeles, making it difficult to get out of the city. However, WBGO was playing some cool jazz, and the weather was cool, making it rather pleasant to be making it out of the city on this Friday afternoon. Nat King Cole was singing as Jo and Maria slowly made their way out of LA and drove toward Barstow. They planned to get to Las Vegas early enough in the evening to have a nice dinner and go see a show."

doc = Document(content=text)

# Ask a question
query = "Who was singing?"
result = reader.run(query=query, documents=[doc])

top_answer = result["answers"][0]
if top_answer.data:
    print(top_answer.data, top_answer.score)
else:
    print("No answer found")


Nat King Cole 0.834891140460968


In [17]:
# # Some questions that "work":
# reader.predict(query="Where is Pioneer Boulevard located?", documents=[doc])

result = reader.run(query="Where is Pioneer Boulevard located?", documents=[doc])

top_answer = result["answers"][0]
if top_answer.data:
    print(top_answer.data, top_answer.score)
else:
    print("No answer found")

Los Angeles 0.7540712356567383


In [ ]:
# reader.predict(query="Who drove to Las Vegas?", documents=[doc])

result = reader.run(query="Who drove to Las Vegas?", documents=[doc])

top_answer = result["answers"][0]
if top_answer.data:
    print(top_answer.data, top_answer.score)
else:
    print("No answer found")


[ExtractedAnswer(query='Who drove to Las Vegas?', score=0.7595530152320862, data='Jo and Maria', document=Document(id=56c54105d21661a451ab2417390549b79e399a12ef8a16c70b4334a81b893526, content: 'The traffic began to slow down on Pioneer Boulevard in Los Angeles, making it difficult to get out o...'), context=None, document_offset=ExtractedAnswer.Span(start=293, end=305), context_offset=None, meta={}), ExtractedAnswer(query='Who drove to Las Vegas?', score=0.24044698476791382, data=None, document=None, context=None, document_offset=None, context_offset=None, meta={})]


In [ ]:
# reader.predict(query="Who is singing?", documents=[doc])

result = reader.run(query="Who is singing?", documents=[doc])

top_answer = result["answers"][0]
if top_answer.data:
    print(top_answer.data, top_answer.score)
else:
    print("No answer found")


[ExtractedAnswer(query='Who is singing?', score=0.8331555724143982, data='Nat King Cole', document=Document(id=56c54105d21661a451ab2417390549b79e399a12ef8a16c70b4334a81b893526, content: 'The traffic began to slow down on Pioneer Boulevard in Los Angeles, making it difficult to get out o...'), context=None, document_offset=ExtractedAnswer.Span(start=264, end=277), context_offset=None, meta={}), ExtractedAnswer(query='Who is singing?', score=0.1668444275856018, data=None, document=None, context=None, document_offset=None, context_offset=None, meta={})]


In [ ]:
# reader.predict(query="What is the plan for the night?", documents=[doc])

result = reader.run(query="What is the plan for the night?", documents=[doc])

top_answer = result["answers"][0]
if top_answer.data:
    print(top_answer.data, top_answer.score)
else:
    print("No answer found")


[ExtractedAnswer(query='What is the plan for the night?', score=0.6904204487800598, data='They planned to get to Las Vegas early enough in the evening to have a nice dinner and go see a show', document=Document(id=56c54105d21661a451ab2417390549b79e399a12ef8a16c70b4334a81b893526, content: 'The traffic began to slow down on Pioneer Boulevard in Los Angeles, making it difficult to get out o...'), context=None, document_offset=ExtractedAnswer.Span(start=364, end=464), context_offset=None, meta={}), ExtractedAnswer(query='What is the plan for the night?', score=0.3095795512199402, data=None, document=None, context=None, document_offset=None, context_offset=None, meta={})]


In [18]:
# Some questions where the answer is not in the text (and the model therefore cannot find it)
# If you inspect the results, you will see that the value "no_ans_gap" is negative for all these questions and actually indicates that the likelihood of "no answer" is higher than the best textual answer
# questions = ["Where is Los Angeles located?","Where is LA located?","Where is Barstow located?","Where is Las Vegas located ?"]
# for q in questions:
#   result = reader.predict(query=q, documents=[doc])
#   print(result)
#   print("\n")

questions = ["Where is Los Angeles located?", "Where is LA located?", "Where is Barstow located?", "Where is Las Vegas located ?"]
for q in questions:
    result = reader.run(query=q, documents=[doc])
    top_answer = result["answers"][0]
    if top_answer.data:
        print(top_answer.data, top_answer.score)
    else:
        print("No answer found")
    print("\n")

Pioneer Boulevard in Los Angeles, making it difficult to get out of the city. However, WBGO was playing some cool jazz, and the weather was cool, making it rather pleasant to be making it out of the city on this Friday afternoon. Nat King Cole was singing as Jo and Maria slowly made their way out of LA and drove toward Barstow 0.5267311334609985


Pioneer Boulevard in Los Angeles, making it difficult to get out of the city. However, WBGO was playing some cool jazz, and the weather was cool, making it rather pleasant to be making it out of the city on this Friday afternoon. Nat King Cole was singing as Jo and Maria slowly made their way out of LA and drove toward Barstow 0.5404552221298218


Las Vegas 0.5181455016136169


Los Angeles 0.49936649203300476




In [20]:
# We can also directly make use of this "no answer" option and allow our reader to return "no answer" (indicated via "answer: None" in the results) by enabling the arg in the FARMreader:
# reader = FARMReader(model_name_or_path="deepset/roberta-base-squad2", use_gpu=True, no_ans_boost=0, return_no_answer=True)
# for q in questions:
#   result = reader.predict(query=q, documents=[doc])
#   print(result)
#   print("\n")

reader = ExtractiveReader(model="deepset/roberta-base-squad2", no_answer=True)
reader.warm_up()

for q in questions:
    result = reader.run(query=q, documents=[doc])
    top_answer = result["answers"][0]
    if top_answer.data:
        print(top_answer.data, top_answer.score)
    else:
        print("No answer found")

    print("\n")

Pioneer Boulevard in Los Angeles, making it difficult to get out of the city. However, WBGO was playing some cool jazz, and the weather was cool, making it rather pleasant to be making it out of the city on this Friday afternoon. Nat King Cole was singing as Jo and Maria slowly made their way out of LA and drove toward Barstow 0.5267311334609985


Pioneer Boulevard in Los Angeles, making it difficult to get out of the city. However, WBGO was playing some cool jazz, and the weather was cool, making it rather pleasant to be making it out of the city on this Friday afternoon. Nat King Cole was singing as Jo and Maria slowly made their way out of LA and drove toward Barstow 0.5404552221298218


Las Vegas 0.5181455016136169


Los Angeles 0.49936649203300476


